In [1]:
# ============================================================
# PRÉDICTION DU TAUX DE RÉUSSITE - VALIDATION CROISÉE
# ============================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error

print("="*60)
print("VALIDATION CROISÉE - STABILITÉ DU MODÈLE")
print("="*60)

# ============================================================
# 1. CHARGEMENT
# ============================================================
print("\n1. CHARGEMENT...")
df_diplomes = pd.read_csv('../data/fact_diplomes.csv', sep=';', encoding='utf-8')
df_inscrits = pd.read_csv('../data/fact_inscrits.csv', sep=';', encoding='utf-8')

# ============================================================
# 2. NETTOYAGE
# ============================================================
df_diplomes = df_diplomes.drop_duplicates().dropna()
df_inscrits = df_inscrits.drop_duplicates().dropna()

# ============================================================
# 3. AGRÉGATION
# ============================================================
df_diplomes_agg = df_diplomes.groupby(['universite_code','etablissement_code','domain_code']).agg({
    'diplomes_M':'sum','diplomes_F':'sum','diplomes_total':'sum'
}).reset_index()

df_inscrits_2022 = df_inscrits[df_inscrits['annee'] == 2022].copy()
df_inscrits_2022['inscrits_total'] = df_inscrits_2022['inscrits_f'] + df_inscrits_2022['inscrits_m']
df_inscrits_agg = df_inscrits_2022.groupby(['code_universite','code_etablissement','code_domaine']).agg({
    'inscrits_f':'sum','inscrits_m':'sum','inscrits_total':'sum'
}).reset_index()

# ============================================================
# 4. FUSION
# ============================================================
df_diplomes_agg = df_diplomes_agg.rename(columns={'universite_code':'univ_code','etablissement_code':'etab_code'})
df_inscrits_agg = df_inscrits_agg.rename(columns={'code_universite':'univ_code','code_etablissement':'etab_code','code_domaine':'domain_code'})
df = pd.merge(df_diplomes_agg, df_inscrits_agg, on=['univ_code','etab_code','domain_code'], how='inner')

# ============================================================
# 5. CIBLE ET FEATURES
# ============================================================
df['taux_reussite'] = (df['diplomes_total'] / df['inscrits_total']).clip(0, 1)
df['ratio_feminisation'] = df['inscrits_f'] / df['inscrits_total']
df['log_taille'] = np.log1p(df['inscrits_total'])
df['efficacite_F'] = df['diplomes_F'] / (df['inscrits_f'] + 1)
df['efficacite_M'] = df['diplomes_M'] / (df['inscrits_m'] + 1)
df['ecart_genre'] = abs(df['efficacite_F'] - df['efficacite_M'])

features = ['ratio_feminisation', 'log_taille', 'efficacite_F', 'efficacite_M', 'ecart_genre']
X = df[features]
y = df['taux_reussite']

# ============================================================
# 6. NORMALISATION
# ============================================================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ============================================================
# 7. VALIDATION CROISÉE RANDOM FOREST
# ============================================================
print("\n7. VALIDATION CROISÉE RANDOM FOREST...")

rf = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)

# Validation croisée 5 folds
cv_scores_rf = cross_val_score(rf, X_scaled, y, cv=5, scoring='r2')
print(f"   Scores par fold: {cv_scores_rf}")
print(f"   Moyenne: {cv_scores_rf.mean():.4f}")
print(f"   Écart-type: {cv_scores_rf.std():.4f}")

# ============================================================
# 8. VALIDATION CROISÉE GRADIENT BOOSTING
# ============================================================
print("\n8. VALIDATION CROISÉE GRADIENT BOOSTING...")

gb = GradientBoostingRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, random_state=42)

cv_scores_gb = cross_val_score(gb, X_scaled, y, cv=5, scoring='r2')
print(f"   Scores par fold: {cv_scores_gb}")
print(f"   Moyenne: {cv_scores_gb.mean():.4f}")
print(f"   Écart-type: {cv_scores_gb.std():.4f}")

# ============================================================
# 9. COMPARAISON
# ============================================================
print("\n" + "="*60)
print("COMPARAISON DE STABILITÉ")
print("="*60)

print(f"\nRandom Forest:")
print(f"   R² moyen (CV): {cv_scores_rf.mean():.4f} ± {cv_scores_rf.std():.4f}")

print(f"\nGradient Boosting:")
print(f"   R² moyen (CV): {cv_scores_gb.mean():.4f} ± {cv_scores_gb.std():.4f}")

if cv_scores_gb.std() < cv_scores_rf.std():
    print("\n✅ Gradient Boosting est plus stable")
else:
    print("\n✅ Random Forest est plus stable")

print("\n" + "="*60)
print("VALIDATION CROISÉE TERMINÉE")
print("="*60)

VALIDATION CROISÉE - STABILITÉ DU MODÈLE

1. CHARGEMENT...

7. VALIDATION CROISÉE RANDOM FOREST...
   Scores par fold: [0.93320196 0.98456832 0.9152812  0.98781063 0.61323357]
   Moyenne: 0.8868
   Écart-type: 0.1397

8. VALIDATION CROISÉE GRADIENT BOOSTING...
   Scores par fold: [0.91710696 0.97994106 0.84728903 0.99229206 0.75990198]
   Moyenne: 0.8993
   Écart-type: 0.0867

COMPARAISON DE STABILITÉ

Random Forest:
   R² moyen (CV): 0.8868 ± 0.1397

Gradient Boosting:
   R² moyen (CV): 0.8993 ± 0.0867

✅ Gradient Boosting est plus stable

VALIDATION CROISÉE TERMINÉE
